In [3]:
!pip install ultralytics
!pip install kaggle
!pip install opendatasets

In [ ]:
import kagglehub

path = kagglehub.dataset_download("fareselmenshawii/face-detection-dataset")

print("Path to dataset files:", path)

c:\Users\uliss\projetos\bootcamp-avanti-ml-proj1-deteccao-facial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\uliss\.cache\kagglehub\datasets\fareselmenshawii\face-detection-dataset\versions\3


In [5]:
!pip install PyYAML

In [6]:
from ultralytics import YOLO
import yaml
import os

# O diretório do dataset baixado
img_dir = path #"/root/.cache/kagglehub/datasets/fareselmenshawii/face-detection-dataset/versions/3"

data_config = {
    "train": os.path.join(path, "images", "train"),
    "val": os.path.join(path, "images", "val"),
    "nc": 1,
    "names": ["face"]
}

yaml_path = "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"Arquivo configurado com sucesso em: {yaml_path}")

Arquivo configurado com sucesso em: data.yaml


In [ ]:
with open("data.yaml", "r") as f:
    loaded_data = yaml.safe_load(f)

print(loaded_data)

{'names': ['face'], 'nc': 1, 'train': 'C:\\Users\\uliss\\.cache\\kagglehub\\datasets\\fareselmenshawii\\face-detection-dataset\\versions\\3\\images\\train', 'val': 'C:\\Users\\uliss\\.cache\\kagglehub\\datasets\\fareselmenshawii\\face-detection-dataset\\versions\\3\\images\\val'}


In [ ]:
import torch

model = YOLO("yolo11n.pt")

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

results = model.train(
    data=yaml_path,
    epochs=20,
    imgsz=640,
    device=device
)

Usando dispositivo: 0
Ultralytics 8.4.21  Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train7, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [11]:
!pip install pandas
import pandas as pd

In [12]:
print(results)

csv_path = os.path.join(results.save_dir, "results.csv")

if os.path.exists(csv_path):
    results_csv = pd.read_csv(csv_path)
    print("Resultados carregados com sucesso!")
    print(results_csv.head())
else:
    print(f"Erro: O arquivo não foi encontrado em {csv_path}")
    # Caso você tenha reiniciado o kernel, tente o caminho padrão:
    # results_csv = pd.read_csv("runs/detect/train/results.csv")

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000016618852120>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [13]:
results_csv.columns

Index(['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
       'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)',
       'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss',
       'lr/pg0', 'lr/pg1', 'lr/pg2'],
      dtype='str')

In [ ]:
!pip install scikit-learn

import cv2
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

# Funções auxiliares
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / (boxAArea + boxBArea - interArea)
    return iou

def load_ground_truth_boxes(label_path, image_shape):
    h, w = image_shape[:2]
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f.readlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
            boxes.append((x1, y1, x2, y2))
    return boxes

# ====== DEFININDO OS CAMINHOS ======
# A variável 'path' já existe no seu notebook (do download do dataset)
# Verifique se ela está definida; caso contrário, use o caminho absoluto.
if 'path' not in dir():
    path = kagglehub.dataset_download("fareselmenshawii/face-detection-dataset")
    print("Caminho do dataset definido como:", path)
else:
    print("Usando path existente:", path)

val_images_dir = os.path.join(path, "images", "val")
val_labels_dir = os.path.join(path, "labels", "val")

if not os.path.exists(val_images_dir):
    raise FileNotFoundError(f"Diretório de imagens de validação não encontrado: {val_images_dir}")
if not os.path.exists(val_labels_dir):
    raise FileNotFoundError(f"Diretório de labels de validação não encontrado: {val_labels_dir}")

val_images = [f for f in os.listdir(val_images_dir) 
              if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
print(f"Total de imagens de validação: {len(val_images)}")

model = YOLO('runs/detect/train7/weights/best.pt') 

# Limiares
CONF_THRESHOLD = 0.25   # confiança mínima
IOU_THRESHOLD = 0.5     # IoU mínimo para match

# Acumuladores
y_true = []   # 1 = face, 0 = fundo (ground truth)
y_pred = []   # 1 = modelo disse face, 0 = modelo disse fundo

for img_name in val_images:
    img_path = os.path.join(val_images_dir, img_name)
    label_path = os.path.join(val_labels_dir, img_name.rsplit('.', 1)[0] + '.txt')

    # Predição com YOLO
    results = model(img_path, conf=CONF_THRESHOLD, verbose=False)[0]

    # Extrair bounding boxes preditas
    pred_boxes = []
    if results.boxes is not None:
        for box in results.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            pred_boxes.append((x1, y1, x2, y2))

    # Carregar ground truth
    img = cv2.imread(img_path)
    if img is None:
        print(f"Erro ao carregar imagem: {img_path}")
        continue
    gt_boxes = load_ground_truth_boxes(label_path, img.shape)

    # Matching
    matched = set()
    for pred in pred_boxes:
        match = False
        for i, gt in enumerate(gt_boxes):
            if i in matched:
                continue
            iou = compute_iou(pred, gt)
            if iou >= IOU_THRESHOLD:
                y_true.append(1)
                y_pred.append(1)
                matched.add(i)
                match = True
                break
        if not match:
            y_true.append(0)
            y_pred.append(1)   # FP

    # Ground truths não detectadas → FN
    for i in range(len(gt_boxes)):
        if i not in matched:
            y_true.append(1)
            y_pred.append(0)

# Cálculo das métricas
print("\n" + "="*60)
print("Métricas de detecção (YOLO) - mesmo estilo HOG+SVM")
print("="*60)
print(f"Total de amostras (caixas consideradas): {len(y_true)}")
print(f"Verdadeiros Positivos (TP): {sum((np.array(y_true)==1) & (np.array(y_pred)==1))}")
print(f"Falsos Positivos (FP)     : {sum((np.array(y_true)==0) & (np.array(y_pred)==1))}")
print(f"Falsos Negativos (FN)     : {sum((np.array(y_true)==1) & (np.array(y_pred)==0))}")
print(f"Verdadeiros Negativos (TN): {sum((np.array(y_true)==0) & (np.array(y_pred)==0))}")
print("\n")
print(f"Precisão: {precision_score(y_true, y_pred):.4f}")
print(f"Recall  : {recall_score(y_true, y_pred):.4f}")
print(f"F1-score: {f1_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusão:")
print(confusion_matrix(y_true, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=['fundo', 'face']))

Usando path existente: C:\Users\uliss\.cache\kagglehub\datasets\fareselmenshawii\face-detection-dataset\versions\3
Total de imagens de validação: 3347

Métricas de detecção (YOLO) - mesmo estilo HOG+SVM
Total de amostras (caixas consideradas): 11586
Verdadeiros Positivos (TP): 8424
Falsos Positivos (FP)     : 1287
Falsos Negativos (FN)     : 1875
Verdadeiros Negativos (TN): 0


Precisão: 0.8675
Recall  : 0.8179
F1-score: 0.8420

Matriz de Confusão:
[[   0 1287]
 [1875 8424]]

Relatório de Classificação:
              precision    recall  f1-score   support

       fundo       0.00      0.00      0.00      1287
        face       0.87      0.82      0.84     10299

    accuracy                           0.73     11586
   macro avg       0.43      0.41      0.42     11586
weighted avg       0.77      0.73      0.75     11586



In [ ]:
thresholds = [0.1, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results_summary = []

for conf in thresholds:
    y_true = []
    y_pred = []
    for img_name in val_images:
        img_path = os.path.join(val_images_dir, img_name)
        label_path = os.path.join(val_labels_dir, img_name.rsplit('.', 1)[0] + '.txt')
        results = model(img_path, conf=conf, verbose=False)[0]
        pred_boxes = []
        if results.boxes is not None:
            for box in results.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)
                pred_boxes.append((x1, y1, x2, y2))
        img = cv2.imread(img_path)
        gt_boxes = load_ground_truth_boxes(label_path, img.shape)
        matched = set()
        for pred in pred_boxes:
            match = False
            for i, gt in enumerate(gt_boxes):
                if i in matched:
                    continue
                iou = compute_iou(pred, gt)
                if iou >= IOU_THRESHOLD:
                    y_true.append(1)
                    y_pred.append(1)
                    matched.add(i)
                    match = True
                    break
            if not match:
                y_true.append(0)
                y_pred.append(1)
        for i in range(len(gt_boxes)):
            if i not in matched:
                y_true.append(1)
                y_pred.append(0)
    
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    results_summary.append((conf, prec, rec, f1))
    print(f"conf={conf:.2f} -> Precisão={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")

print("\nResumo:")
for conf, prec, rec, f1 in results_summary:
    print(f"conf={conf:.2f}: prec={prec:.4f}, rec={rec:.4f}, f1={f1:.4f}")

conf=0.10 -> Precisão=0.7168, Recall=0.8636, F1=0.7834
conf=0.25 -> Precisão=0.8675, Recall=0.8179, F1=0.8420
conf=0.30 -> Precisão=0.8910, Recall=0.8031, F1=0.8448
conf=0.40 -> Precisão=0.9228, Recall=0.7738, F1=0.8417
conf=0.50 -> Precisão=0.9457, Recall=0.7346, F1=0.8269
conf=0.60 -> Precisão=0.9660, Recall=0.6839, F1=0.8008
conf=0.70 -> Precisão=0.9839, Recall=0.6000, F1=0.7454
conf=0.80 -> Precisão=0.9920, Recall=0.3975, F1=0.5676
conf=0.90 -> Precisão=1.0000, Recall=0.0240, F1=0.0468

Resumo:
conf=0.10: prec=0.7168, rec=0.8636, f1=0.7834
conf=0.25: prec=0.8675, rec=0.8179, f1=0.8420
conf=0.30: prec=0.8910, rec=0.8031, f1=0.8448
conf=0.40: prec=0.9228, rec=0.7738, f1=0.8417
conf=0.50: prec=0.9457, rec=0.7346, f1=0.8269
conf=0.60: prec=0.9660, rec=0.6839, f1=0.8008
conf=0.70: prec=0.9839, rec=0.6000, f1=0.7454
conf=0.80: prec=0.9920, rec=0.3975, f1=0.5676
conf=0.90: prec=1.0000, rec=0.0240, f1=0.0468


In [ ]:
thresholds = [0.1, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
IOU_THRESHOLD = 0.5

for conf in thresholds:
    y_true = []
    y_pred = []
    
    print(f"\n{'='*60}")
    print(f"Threshold de confiança: {conf}")
    print('='*60)
    
    for img_name in val_images:
        img_path = os.path.join(val_images_dir, img_name)
        label_path = os.path.join(val_labels_dir, img_name.rsplit('.', 1)[0] + '.txt')
        
        results = model(img_path, conf=conf, verbose=False)[0]
        pred_boxes = []
        if results.boxes is not None:
            for box in results.boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)
                pred_boxes.append((x1, y1, x2, y2))
        
        img = cv2.imread(img_path)
        if img is None:
            continue
        gt_boxes = load_ground_truth_boxes(label_path, img.shape)
        
        matched = set()
        for pred in pred_boxes:
            match = False
            for i, gt in enumerate(gt_boxes):
                if i in matched:
                    continue
                iou = compute_iou(pred, gt)
                if iou >= IOU_THRESHOLD:
                    y_true.append(1)
                    y_pred.append(1)
                    matched.add(i)
                    match = True
                    break
            if not match:
                y_true.append(0)
                y_pred.append(1)
        
        for i in range(len(gt_boxes)):
            if i not in matched:
                y_true.append(1)
                y_pred.append(0)
    
    # Cálculo das métricas
    print(f"\nTotal de amostras: {len(y_true)}")
    print(f"TP: {sum((np.array(y_true)==1) & (np.array(y_pred)==1))}")
    print(f"FP: {sum((np.array(y_true)==0) & (np.array(y_pred)==1))}")
    print(f"FN: {sum((np.array(y_true)==1) & (np.array(y_pred)==0))}")
    print(f"TN: {sum((np.array(y_true)==0) & (np.array(y_pred)==0))}")
    
    print("\nMatriz de Confusão:")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    
    print("\nRelatório de Classificação:")
    print(classification_report(y_true, y_pred, target_names=['fundo', 'face']))
    
    # (Opcional) Exibir também as métricas isoladas
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"Precisão: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")


Threshold de confiança: 0.1

Total de amostras: 13813
TP: 8894
FP: 3514
FN: 1405
TN: 0

Matriz de Confusão:
[[   0 3514]
 [1405 8894]]

Relatório de Classificação:
              precision    recall  f1-score   support

       fundo       0.00      0.00      0.00      3514
        face       0.72      0.86      0.78     10299

    accuracy                           0.64     13813
   macro avg       0.36      0.43      0.39     13813
weighted avg       0.53      0.64      0.58     13813

Precisão: 0.7168, Recall: 0.8636, F1: 0.7834

Threshold de confiança: 0.25

Total de amostras: 11586
TP: 8424
FP: 1287
FN: 1875
TN: 0

Matriz de Confusão:
[[   0 1287]
 [1875 8424]]

Relatório de Classificação:
              precision    recall  f1-score   support

       fundo       0.00      0.00      0.00      1287
        face       0.87      0.82      0.84     10299

    accuracy                           0.73     11586
   macro avg       0.43      0.41      0.42     11586
weighted avg       0.77  

c:\Users\uliss\projetos\bootcamp-avanti-ml-proj1-deteccao-facial\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\uliss\projetos\bootcamp-avanti-ml-proj1-deteccao-facial\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\uliss\projetos\bootcamp-avanti-ml-proj1-deteccao-facial\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this b

: 